<a href="https://colab.research.google.com/github/jeremy26/hydranets_course/blob/claude/modernize-autoware-course-edY16/Module_2_Depth_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: HydraNet — Drivable Area + Depth

In this module, you'll build a **modern HydraNet** that performs **drivable area segmentation** and **monocular depth estimation** simultaneously from a single camera image.

**Architecture** (based on [Autoware Vision Pilot](https://github.com/autowarefoundation/autoware_vision_pilot)):

```
Image -> EfficientNet-B0 Backbone -> Context Module -> Shared Neck -> Drivable Head
                                                                   -> Depth Head
```

**Key concepts:**
- Shared backbone with multi-scale features
- Context modules for global scene understanding (pseudo-attention)
- U-Net style decoder neck with skip connections
- Task-specific lightweight heads
- Multi-task loss balancing

**Dataset:** BDD100K 10k curated subset — drivable area labels + Depth Anything v2 pseudo-depth

# 1 — Setup & Installation

In [ ]:
# Clone the course repo and install dependencies
!git clone -b claude/modernize-autoware-course-edY16 https://github.com/jeremy26/hydranets_course.git 2>/dev/null || true
%cd hydranets_course

!pip install -q torchvision pillow matplotlib numpy tqdm

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Mount Google Drive to save models
from google.colab import drive
drive.mount('/content/gdrive')
print('Google Drive mounted at /content/gdrive')

# 2 — Dataset: BDD100K + Pseudo-Depth

We use a curated **10k subset of BDD100K** with two label types for every image:

- **Drivable area masks** — 3 classes: background, directly drivable (ego lane), alternatively drivable (adjacent lanes)
- **Pseudo-depth maps** — generated by [Depth Anything v2](https://github.com/DepthAnything/Depth-Anything-V2), a state-of-the-art monocular depth foundation model

Using drivable area (vs semantic segmentation) gives us **~10k labeled images** instead of ~3k — more data, better results, and labels that are directly useful for autonomous driving.

The dataset was pre-processed and is hosted on S3 with a flat, clean structure:
```
bdd_hydranet_10k/
  train/  images/ depth/ drivable/ lanes/
  val/    images/ depth/ drivable/ lanes/
  test/   images/ depth/ drivable/ lanes/
  labels_train.json    ← full BDD100K annotations for Module 3
```

In [ ]:
import os

# Download the pre-processed dataset (run once)
# Uncomment the line below after replacing [URL] with your S3 URL:
# !wget -q [URL]/bdd_hydranet_10k.zip && unzip -q bdd_hydranet_10k.zip

# For the full 100k dataset (better results, larger download):
# !wget -q [URL]/bdd_hydranet_full.zip && unzip -q bdd_hydranet_full.zip

print("Dataset ready!" if os.path.exists("bdd_hydranet_10k") else "Run the wget above first!")

In [ ]:
import os

DATA_ROOT = "bdd_hydranet_10k"   # swap to "bdd_hydranet_full" for 100k

# Verify
for split in ['train', 'val']:
    for sub in ['images', 'depth', 'drivable', 'lanes']:
        path = os.path.join(DATA_ROOT, split, sub)
        count = len(os.listdir(path)) if os.path.isdir(path) else 0
        status = f"{count} files" if count else "NOT FOUND"
        print(f"  {split}/{sub}: {status}")

## Visualize the Data

Let's look at a few examples: RGB image, drivable area mask, and pseudo-depth. Every image in the dataset has all three labels — no missing data.

In [ ]:
# Drivable area color palette (3 classes)
DRIVABLE_COLORS = np.array([
    [  0,   0,   0],   # 0 = background (not drivable)
    [  0, 200,   0],   # 1 = direct (ego lane — you can drive here)
    [  0, 100, 200],   # 2 = alternative (adjacent drivable lanes)
], dtype=np.uint8)

DRIVABLE_CLASSES = ['background', 'direct (ego lane)', 'alternative']

def colorize_drivable(mask):
    """Convert a 3-class drivable area mask to an RGB image."""
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, color in enumerate(DRIVABLE_COLORS):
        rgb[mask == cls_id] = color
    return rgb

In [ ]:
# Visualize samples — every image has all labels, no filtering needed
img_dir      = os.path.join(DATA_ROOT, 'train', 'images')
drivable_dir = os.path.join(DATA_ROOT, 'train', 'drivable')
depth_dir    = os.path.join(DATA_ROOT, 'train', 'depth')

images = sorted(os.listdir(img_dir))
print(f"Training images: {len(images)}")

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
for i in range(3):
    name = images[np.random.randint(0, len(images))]
    stem = os.path.splitext(name)[0]

    img = Image.open(os.path.join(img_dir, name))
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('RGB Image')
    axes[i, 0].axis('off')

    drivable = np.array(Image.open(os.path.join(drivable_dir, stem + '.png')))
    axes[i, 1].imshow(colorize_drivable(drivable))
    axes[i, 1].set_title('Drivable Area (green=ego, blue=adjacent)')
    axes[i, 1].axis('off')

    depth = np.array(Image.open(os.path.join(depth_dir, stem + '.jpg')).convert('L'))
    axes[i, 2].imshow(depth, cmap='magma')
    axes[i, 2].set_title('Pseudo-Depth (Depth Anything v2)')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## DataLoader

Simple, flat dataset class — one `os.listdir`, no searching. Every image has matching drivable + depth labels by construction.

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ImageNet normalization (used by EfficientNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

INPUT_SIZE = (320, 640)  # (H, W)


class BDD100KDataset(Dataset):
    """BDD100K multi-task dataset.

    Loads from the clean flat structure produced by prepare_dataset.py:
        root/split/images/{id}.jpg
        root/split/drivable/{id}.png
        root/split/depth/{id}.jpg
    """

    def __init__(self, root, split='train'):
        self.img_dir      = os.path.join(root, split, 'images')
        self.drivable_dir = os.path.join(root, split, 'drivable')
        self.depth_dir    = os.path.join(root, split, 'depth')
        self.filenames    = sorted(os.listdir(self.img_dir))

        self.img_transform = transforms.Compose([
            transforms.Resize(INPUT_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        stem = os.path.splitext(name)[0]

        # Image
        image = Image.open(os.path.join(self.img_dir, name)).convert('RGB')
        image = self.img_transform(image)

        # Drivable area mask
        drivable = Image.open(os.path.join(self.drivable_dir, stem + '.png'))
        drivable = drivable.resize((INPUT_SIZE[1], INPUT_SIZE[0]), Image.NEAREST)
        drivable = torch.from_numpy(np.array(drivable)).long()

        # Depth map
        depth = Image.open(os.path.join(self.depth_dir, stem + '.jpg')).convert('L')
        depth = depth.resize((INPUT_SIZE[1], INPUT_SIZE[0]), Image.BILINEAR)
        depth = torch.from_numpy(np.array(depth, dtype=np.float32)).unsqueeze(0) / 255.0

        return {'image': image, 'drivable': drivable, 'depth': depth, 'filename': stem}


train_dataset = BDD100KDataset(DATA_ROOT, split='train')
val_dataset   = BDD100KDataset(DATA_ROOT, split='val')

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} images | Val: {len(val_dataset)} images")
print(f"Input: {INPUT_SIZE}")

batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  image:    {list(batch['image'].shape)}")
print(f"  drivable: {list(batch['drivable'].shape)}  (0=bg, 1=direct, 2=alternative)")
print(f"  depth:    {list(batch['depth'].shape)}")

# 3 — Architecture: Building the HydraNet

Our HydraNet follows the **Autoware Vision Pilot** pattern. Every task goes through 4 stages:

```
Image ─> [Backbone] ─> [Context] ─> [Neck] ─> [Head] ─> Prediction
              │                        ▲
              └── skip connections ─────┘
```

| Component | Role | Trainable? |
|-----------|------|-----------|
| **Backbone** | EfficientNet-B0 — extracts multi-scale features | Pretrained, fine-tuned |
| **Context** | Global Average Pool → MLP → spatial reconstruction → pseudo-attention | Yes |
| **Neck** | U-Net decoder with skip connections from backbone | Yes (shared) |
| **Head** | Lightweight task-specific output layers | Yes (per-task) |

Let's build each component step by step.

## 3.1 — Backbone: EfficientNet-B0

We use **EfficientNet-B0** (pretrained on ImageNet) as our backbone encoder. Unlike the old MobileNetv2, EfficientNet uses compound scaling and is more efficient.

The backbone extracts features at **5 different scales** — these multi-scale features are crucial for the skip connections in the decoder.

In [ ]:
from torchvision import models


class Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super(Backbone, self).__init__()
        if pretrained:
            self.encoder = models.efficientnet_b0(
                weights='EfficientNet_B0_Weights.IMAGENET1K_V1'
            ).features
        else:
            self.encoder = models.efficientnet_b0(weights=None).features

    def forward(self, image):
        # EfficientNet-B0 feature stages — strides shown for INPUT_SIZE=(256,512)
        l0 = self.encoder[0](image)   # (B,  32, H/2,  W/2)   = (B,  32, 128, 256)
        l1 = self.encoder[1](l0)      # (B,  16, H/2,  W/2)
        l2 = self.encoder[2](l1)      # (B,  24, H/4,  W/4)   = (B,  24,  64, 128)
        l3 = self.encoder[3](l2)      # (B,  40, H/8,  W/8)   = (B,  40,  32,  64)
        l4 = self.encoder[4](l3)      # (B,  80, H/16, W/16)  = (B,  80,  16,  32)
        l5 = self.encoder[5](l4)      # (B, 112, H/16, W/16)
        l6 = self.encoder[6](l5)      # (B, 192, H/32, W/32)  = (B, 192,   8,  16)
        l7 = self.encoder[7](l6)      # (B, 320, H/32, W/32)
        l8 = self.encoder[8](l7)      # (B,1280, H/32, W/32)  = (B,1280,   8,  16)
        return [l0, l2, l3, l4, l8]


backbone = Backbone(pretrained=True)

dummy_input = torch.randn(1, 3, *INPUT_SIZE)
features = backbone(dummy_input)

print("Backbone multi-scale features:")
print(f"{'Level':<10} {'Shape':<30} {'Use'}")
print("-" * 65)
for i, f in enumerate(features):
    use = ['skip to head (H/2)', 'skip to neck block 3 (H/4)',
           'skip to neck block 2 (H/8)', 'skip to neck block 1 (H/16)',
           'deep features -> context (H/32)'][i]
    print(f"feat[{i}]    {str(list(f.shape)):<30} {use}")

## 3.2 — Context Module: Global Scene Understanding

The Context module is what makes this architecture special. Before decoding, we ask: **"What kind of scene am I looking at?"**

It works in 3 steps:
1. **Global Average Pooling** — compress the entire feature map to a single vector
2. **MLP** — process through fully connected layers to capture scene-level patterns
3. **Spatial Reconstruction** — reshape back to a spatial feature map
4. **Pseudo-Attention** — `context * features + features` (like a residual gate)

We have two context modules — one for drivable area, one for depth — allowing each task to attend to different scene-level patterns.

In [ ]:
class SceneContext(nn.Module):
    """Context module — compresses deep features, reconstructs as pseudo-attention.

    The Linear bottleneck outputs H/32 * W/32 values (computed from INPUT_SIZE),
    which are reshaped back to match the feature map spatial size dynamically.
    """

    def __init__(self):
        super(SceneContext, self).__init__()
        self.GeLU = nn.GELU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.25)

        # MLP: 1280 -> 800 -> 800 -> H/32*W/32 (dynamic from INPUT_SIZE)
        spatial_size = (INPUT_SIZE[0] // 32) * (INPUT_SIZE[1] // 32)
        self.context_layer_0 = nn.Linear(1280, 800)
        self.context_layer_1 = nn.Linear(800, 800)
        self.context_layer_2 = nn.Linear(800, spatial_size)

        # Spatial reconstruction: (B,1,H/32,W/32) -> (B,1280,H/32,W/32)
        self.context_layer_3 = nn.Conv2d(1, 128, 3, 1, 1)
        self.context_layer_4 = nn.Conv2d(128, 256, 3, 1, 1)
        self.context_layer_5 = nn.Conv2d(256, 512, 3, 1, 1)
        self.context_layer_6 = nn.Conv2d(512, 1280, 3, 1, 1)

    def forward(self, features):
        b, c, h, w = features.shape                          # e.g. (B,1280,8,16)
        feature_vector = torch.mean(features, dim=[2, 3])    # (B,1280)
        c0 = self.GeLU(self.dropout(self.context_layer_0(feature_vector)))
        c1 = self.GeLU(self.dropout(self.context_layer_1(c0)))
        c2 = self.sigmoid(self.dropout(self.context_layer_2(c1)))
        c3 = c2.view(b, 1, h, w)                            # (B,1,8,16) — dynamic!
        c4 = self.GeLU(self.context_layer_3(c3))
        c5 = self.GeLU(self.context_layer_4(c4))
        c6 = self.GeLU(self.context_layer_5(c5))
        c7 = self.GeLU(self.context_layer_6(c6))
        return c7 * features + features                      # pseudo-attention


class DepthContext(nn.Module):
    """Context module for depth estimation — same structure as SceneContext."""

    def __init__(self):
        super(DepthContext, self).__init__()
        self.GeLU = nn.GELU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.25)

        spatial_size = (INPUT_SIZE[0] // 32) * (INPUT_SIZE[1] // 32)
        self.context_layer_0 = nn.Linear(1280, 800)
        self.context_layer_1 = nn.Linear(800, 800)
        self.context_layer_2 = nn.Linear(800, spatial_size)

        self.context_layer_3 = nn.Conv2d(1, 128, 3, 1, 1)
        self.context_layer_4 = nn.Conv2d(128, 256, 3, 1, 1)
        self.context_layer_5 = nn.Conv2d(256, 512, 3, 1, 1)
        self.context_layer_6 = nn.Conv2d(512, 1280, 3, 1, 1)

    def forward(self, features):
        b, c, h, w = features.shape
        feature_vector = torch.mean(features, dim=[2, 3])
        c0 = self.GeLU(self.dropout(self.context_layer_0(feature_vector)))
        c1 = self.GeLU(self.dropout(self.context_layer_1(c0)))
        c2 = self.sigmoid(self.dropout(self.context_layer_2(c1)))
        c3 = c2.view(b, 1, h, w)
        c4 = self.GeLU(self.context_layer_3(c3))
        c5 = self.GeLU(self.context_layer_4(c4))
        c6 = self.GeLU(self.context_layer_5(c5))
        c7 = self.GeLU(self.context_layer_6(c6))
        return c7 * features + features


drivable_context = SceneContext()
depth_context    = DepthContext()

deep_features = features[4]   # (1, 1280, H/32, W/32)
print(f"Input to context:      {list(deep_features.shape)}")
print(f"Drivable context out:  {list(drivable_context(deep_features).shape)}")
print(f"Depth context out:     {list(depth_context(deep_features).shape)}")
print(f"\nSame shape as input — context acts as a per-channel attention gate.")

## 3.3 — Neck: U-Net Style Decoder

The Neck upsamples the context-enhanced features back to higher resolution, using **skip connections** from the encoder at each stage.

```
context (1280ch, H/32) ──┐
                          ▼
               [ConvTranspose2d 2x]  +  features[3] (80ch, H/16)  ──> 768ch, H/16
                          ▼
               [ConvTranspose2d 2x]  +  features[2] (40ch, H/8)   ──> 512ch, H/8
                          ▼
               [ConvTranspose2d 2x]  +  features[1] (24ch, H/4)   ──> 256ch, H/4 = NECK OUTPUT
```

The neck is **shared** across tasks — both segmentation and depth use the same decoded features.

In [ ]:
class SceneNeck(nn.Module):
    def __init__(self):
        super(SceneNeck, self).__init__()
        self.GeLU = nn.GELU()

        # Upsample block 1: H/32 -> H/16
        self.upsample_layer_0 = nn.ConvTranspose2d(1280, 1280, 2, 2)
        self.skip_link_layer_0 = nn.Conv2d(80, 1280, 1)  # Match features[3] channels
        self.decode_layer_0 = nn.Conv2d(1280, 768, 3, 1, 1)
        self.decode_layer_1 = nn.Conv2d(768, 768, 3, 1, 1)

        # Upsample block 2: H/16 -> H/8
        self.upsample_layer_1 = nn.ConvTranspose2d(768, 768, 2, 2)
        self.skip_link_layer_1 = nn.Conv2d(40, 768, 1)   # Match features[2] channels
        self.decode_layer_2 = nn.Conv2d(768, 512, 3, 1, 1)
        self.decode_layer_3 = nn.Conv2d(512, 512, 3, 1, 1)

        # Upsample block 3: H/8 -> H/4
        self.upsample_layer_2 = nn.ConvTranspose2d(512, 512, 2, 2)
        self.skip_link_layer_2 = nn.Conv2d(24, 512, 1)   # Match features[1] channels
        self.decode_layer_4 = nn.Conv2d(512, 512, 3, 1, 1)
        self.decode_layer_5 = nn.Conv2d(512, 256, 3, 1, 1)

    def forward(self, context, features):
        """
        Args:
            context: (B, 1280, H/32, W/32) from context module
            features: list of encoder features [l0, l2, l3, l4, l8]
                      features[1]=(B,24,H/4), features[2]=(B,40,H/8),
                      features[3]=(B,80,H/16)
        Returns:
            neck: (B, 256, H/4, W/4)
        """
        # Block 1: H/32 -> H/16
        d0 = self.upsample_layer_0(context)
        d0 = d0 + self.skip_link_layer_0(features[3])
        d1 = self.GeLU(self.decode_layer_0(d0))
        d2 = self.GeLU(self.decode_layer_1(d1))

        # Block 2: H/16 -> H/8
        d3 = self.upsample_layer_1(d2)
        d3 = d3 + self.skip_link_layer_1(features[2])
        d3 = self.GeLU(self.decode_layer_2(d3))
        d4 = self.GeLU(self.decode_layer_3(d3))

        # Block 3: H/8 -> H/4
        d5 = self.upsample_layer_2(d4)
        d5 = d5 + self.skip_link_layer_2(features[1])
        d5 = self.GeLU(self.decode_layer_4(d5))
        neck = self.GeLU(self.decode_layer_5(d5))

        return neck


neck_module = SceneNeck()

neck_output = neck_module(drivable_context(deep_features), features)
print(f"Neck output: {list(neck_output.shape)}")
print(f"Resolution: H/4 x W/4 = {INPUT_SIZE[0]//4} x {INPUT_SIZE[1]//4}")
print(f"\nThis 256-channel feature map is what all heads receive.")

## 3.4 — Heads: Task-Specific Output Layers

Heads are **lightweight** — they take the shared neck (256ch, H/4) and upsample to full resolution.

- **DrivableAreaHead** → (B, 3, H, W) — 3-class logits (bg / direct / alternative)
- **DepthHead** → (B, 1, H, W) — depth prediction

Since heads are small, they train fast. This is the exact pattern students use in Module 3 to add new heads!

In [ ]:
class DrivableAreaHead(nn.Module):
    """Drivable area segmentation head.
    Upsamples neck from H/4 to full resolution, predicts 3-class drivable mask.

    Classes: 0=background, 1=direct (ego lane), 2=alternative
    """

    def __init__(self, num_classes=3):
        super(DrivableAreaHead, self).__init__()
        self.GeLU = nn.GELU()

        # H/4 -> H/2
        self.upsample_layer_0  = nn.ConvTranspose2d(256, 256, 2, 2)
        self.skip_link_layer_0 = nn.Conv2d(32, 256, 1)
        self.decode_layer_0    = nn.Conv2d(256, 256, 3, 1, 1)
        self.decode_layer_1    = nn.Conv2d(256, 128, 3, 1, 1)

        # H/2 -> H
        self.upsample_layer_1 = nn.ConvTranspose2d(128, 128, 2, 2)
        self.decode_layer_2   = nn.Conv2d(128, 128, 3, 1, 1)
        self.decode_layer_3   = nn.Conv2d(128, 64, 3, 1, 1)

        self.output_layer = nn.Conv2d(64, num_classes, 3, 1, 1)

    def forward(self, neck, features):
        d0 = self.upsample_layer_0(neck) + self.skip_link_layer_0(features[0])
        d0 = self.GeLU(self.decode_layer_0(d0))
        d1 = self.GeLU(self.decode_layer_1(d0))
        d2 = self.upsample_layer_1(d1)
        d2 = self.GeLU(self.decode_layer_2(d2))
        d3 = self.GeLU(self.decode_layer_3(d2))
        return self.output_layer(d3)


class DepthHead(nn.Module):
    """Monocular depth estimation head. Outputs single-channel depth map."""

    def __init__(self):
        super(DepthHead, self).__init__()
        self.GeLU = nn.GELU()

        self.upsample_layer_0  = nn.ConvTranspose2d(256, 256, 2, 2)
        self.skip_link_layer_0 = nn.Conv2d(32, 256, 1)
        self.decode_layer_0    = nn.Conv2d(256, 256, 3, 1, 1)
        self.decode_layer_1    = nn.Conv2d(256, 128, 3, 1, 1)

        self.upsample_layer_1 = nn.ConvTranspose2d(128, 128, 2, 2)
        self.decode_layer_2   = nn.Conv2d(128, 128, 3, 1, 1)
        self.decode_layer_3   = nn.Conv2d(128, 128, 3, 1, 1)

        self.output_layer = nn.Conv2d(128, 1, 3, 1, 1)

    def forward(self, neck, features):
        d0 = self.upsample_layer_0(neck) + self.skip_link_layer_0(features[0])
        d0 = self.GeLU(self.decode_layer_0(d0))
        d1 = self.GeLU(self.decode_layer_1(d0))
        d2 = self.upsample_layer_1(d1)
        d2 = self.GeLU(self.decode_layer_2(d2))
        d3 = self.GeLU(self.decode_layer_3(d2))
        return self.output_layer(d3)


drivable_head = DrivableAreaHead(num_classes=3)
depth_head    = DepthHead()

drivable_out = drivable_head(neck_output, features)
depth_out    = depth_head(neck_output, features)

print(f"Drivable output: {list(drivable_out.shape)}  (3-class logits at full res)")
print(f"Depth output:    {list(depth_out.shape)}  (1-channel depth at full res)")

## 3.5 — The Full HydraNet

Wires everything together: Backbone → Context (per-task) → Neck (shared) → Heads (per-task).

In [ ]:
torch.cuda.empty_cache()


class HydraNet(nn.Module):
    """Two-task HydraNet: Drivable Area + Depth.

    Backbone (shared) -> Context (per-task) -> Neck (shared) -> Heads (per-task)
    """

    def __init__(self, num_drivable_classes=3):
        super(HydraNet, self).__init__()
        self.backbone       = Backbone(pretrained=True)
        self.drivable_context = SceneContext()
        self.depth_context    = DepthContext()
        self.neck             = SceneNeck()
        self.drivable_head    = DrivableAreaHead(num_classes=num_drivable_classes)
        self.depth_head       = DepthHead()

    def forward(self, image):
        features     = self.backbone(image)
        deep         = features[4]

        driv_ctx     = self.drivable_context(deep)
        driv_neck    = self.neck(driv_ctx, features)
        drivable_out = self.drivable_head(driv_neck, features)

        depth_ctx    = self.depth_context(deep)
        depth_neck   = self.neck(depth_ctx, features)
        depth_out    = self.depth_head(depth_neck, features)

        return drivable_out, depth_out

    def get_backbone_params(self):
        return self.backbone.parameters()

    def get_head_params(self):
        params = []
        for m in [self.drivable_context, self.depth_context,
                  self.neck, self.drivable_head, self.depth_head]:
            params.extend(m.parameters())
        return params


model = HydraNet(num_drivable_classes=3).to(device)

total_params   = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.get_backbone_params())
head_params    = sum(p.numel() for p in model.get_head_params())

print(f"Total parameters:    {total_params:,}")
print(f"Backbone parameters: {backbone_params:,} ({100*backbone_params/total_params:.1f}%)")
print(f"Context+Neck+Heads:  {head_params:,} ({100*head_params/total_params:.1f}%)")

dummy = torch.randn(2, 3, *INPUT_SIZE).to(device)
driv_pred, depth_pred = model(dummy)
print(f"\nForward pass OK!")
print(f"Drivable: {list(driv_pred.shape)}")
print(f"Depth:    {list(depth_pred.shape)}")

# 4 — Loss Functions

- **Drivable Area:** Cross-Entropy Loss (3-class per-pixel classification)
- **Depth:** Inverse Huber (berHu) Loss — L1 for small errors, L2 for large errors
- **Combined:** Weighted sum

In [ ]:
class InverseHuberLoss(nn.Module):
    """Inverse Huber (berHu) loss for depth estimation."""

    def forward(self, prediction, target):
        mask = target > 0
        prediction, target = prediction[mask], target[mask]
        if prediction.numel() == 0:
            return torch.tensor(0.0, device=prediction.device)
        diff = torch.abs(target - prediction)
        c = 0.2 * torch.max(diff).item()
        loss = torch.zeros_like(diff)
        loss[diff <= c] = diff[diff <= c]
        if c > 0:
            loss[diff > c] = (diff[diff > c] ** 2 + c ** 2) / (2 * c)
        return loss.mean()


class DiceLoss(nn.Module):
    """Dice Loss for segmentation — handles class imbalance naturally.
    Directly optimizes per-class overlap, so rare classes get equal gradient."""

    def __init__(self, num_classes=3, smooth=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.softmax(pred, dim=1)
        dice = 0
        for cls in range(self.num_classes):
            p = pred[:, cls].reshape(-1)
            t = (target == cls).float().reshape(-1)
            intersection = (p * t).sum()
            dice += (2 * intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return 1 - dice / self.num_classes


drivable_criterion = DiceLoss(num_classes=3)
depth_criterion    = InverseHuberLoss()

print("Loss functions ready!")
print("  Drivable: DiceLoss (3 classes)")
print("  Depth:    InverseHuberLoss (berHu)")

# 5 — Training

We use **differential learning rates**: a lower LR for the pretrained backbone, and a higher LR for the new layers (context, neck, heads). This prevents destroying the pretrained features while allowing the new layers to learn quickly.

In [ ]:
optimizer = torch.optim.SGD([
    {'params': model.get_backbone_params(), 'lr': 1e-4},
    {'params': model.get_head_params(),     'lr': 1e-2},
], momentum=0.9, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[6, 8], gamma=0.1
)

NUM_EPOCHS = 10
print(f"Training for {NUM_EPOCHS} epochs")
print(f"Backbone LR: {optimizer.param_groups[0]['lr']}")
print(f"Heads LR:    {optimizer.param_groups[1]['lr']}")

In [ ]:
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()


def compute_miou(predictions, targets, num_classes=3, ignore_index=255):
    """Mean IoU for multi-class segmentation."""
    preds = predictions.argmax(dim=1)
    ious = []
    for cls in range(num_classes):
        pred_mask   = (preds == cls)
        target_mask = (targets == cls)
        valid       = (targets != ignore_index)
        intersection = (pred_mask & target_mask & valid).sum().float()
        union        = ((pred_mask | target_mask) & valid).sum().float()
        if union > 0:
            ious.append((intersection / union).item())
    return np.mean(ious) if ious else 0.0


def compute_rmse(prediction, target):
    """RMSE for depth where target > 0."""
    mask = target > 0
    if mask.sum() == 0:
        return 0.0
    diff = prediction[mask] - target[mask]
    return torch.sqrt((diff ** 2).mean()).item()


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    totals = {'drivable': 0, 'depth': 0, 'total': 0}
    n = 0
    for batch in tqdm(loader, desc='Train', leave=False):
        images   = batch['image'].to(device)
        driv_gt  = batch['drivable'].to(device)
        depth_gt = batch['depth'].to(device)

        optimizer.zero_grad()
        with autocast():
            driv_pred, depth_pred = model(images)
            loss_driv  = drivable_criterion(driv_pred, driv_gt)
            loss_depth = depth_criterion(depth_pred, depth_gt)
            total_loss = loss_driv + 2.5 * loss_depth  # weighted: depth ~2.5x smaller scale

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        totals['drivable'] += loss_driv.item()
        totals['depth']    += loss_depth.item()
        totals['total']    += total_loss.item()
        n += 1

    return {k: v / n for k, v in totals.items()}


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    totals = {'drivable': 0, 'depth': 0, 'total': 0}
    total_miou, total_rmse, n = 0, 0, 0

    for batch in tqdm(loader, desc='Val', leave=False):
        images   = batch['image'].to(device)
        driv_gt  = batch['drivable'].to(device)
        depth_gt = batch['depth'].to(device)

        driv_pred, depth_pred = model(images)
        loss_driv  = drivable_criterion(driv_pred, driv_gt)
        loss_depth = depth_criterion(depth_pred, depth_gt)

        totals['drivable'] += loss_driv.item()
        totals['depth']    += loss_depth.item()
        totals['total']    += (loss_driv + 2.5 * loss_depth).item()  # weighted
        total_miou += compute_miou(driv_pred, driv_gt)
        total_rmse += compute_rmse(depth_pred, depth_gt)
        n += 1

    result = {k: v / n for k, v in totals.items()}
    result['miou'] = total_miou / n
    result['rmse'] = total_rmse / n
    return result


print("Training functions defined!")

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_rmse': []}
best_miou = 0.0

for epoch in range(NUM_EPOCHS):
    train_m = train_one_epoch(model, train_loader, optimizer, device)
    val_m   = validate(model, val_loader, device)
    scheduler.step()

    history['train_loss'].append(train_m['total'])
    history['val_loss'].append(val_m['total'])
    history['val_miou'].append(val_m['miou'])
    history['val_rmse'].append(val_m['rmse'])

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"Train {train_m['total']:.4f} (driv:{train_m['drivable']:.4f} dep:{train_m['depth']:.4f}) | "
          f"Val {val_m['total']:.4f} (driv:{val_m['drivable']:.4f} dep:{val_m['depth']:.4f}) | "
          f"mIoU {val_m['miou']:.4f} | "
          f"RMSE {val_m['rmse']:.4f}")
    
    # Debug: print loss ratio first epoch
    if epoch == 0:
        print(f"  → Loss ratio (depth/drivable): {val_m['depth']/val_m['drivable']:.2f}x")

    if val_m['miou'] > best_miou:
        best_miou = val_m['miou']
        torch.save(model.state_dict(), 'hydranet_best.pth')
        print(f"  -> Saved best model (mIoU: {best_miou:.4f})")

print(f"\nTraining complete! Best mIoU: {best_miou:.4f}")

# 6 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['val_miou'], color='green')
axes[1].set_title('Validation mIoU (Drivable Area)')
axes[1].set_xlabel('Epoch')

axes[2].plot(history['val_rmse'], color='orange')
axes[2].set_title('Validation RMSE (Depth)')
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

# 7 — Inference & Visualization

Let's load the best model and visualize predictions on validation images.

In [ ]:
model.load_state_dict(torch.load('hydranet_best.pth', map_location=device))
model.eval()

MEAN = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
STD  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

def denormalize(t):
    return (t.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

val_batch = next(iter(val_loader))
images = val_batch['image'].to(device)

with torch.no_grad():
    driv_pred, depth_pred = model(images)

driv_pred  = driv_pred.argmax(dim=1).cpu().numpy()
depth_pred = depth_pred.squeeze(1).cpu().numpy()

n_show = min(4, images.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(20, 5 * n_show))

for i in range(n_show):
    axes[i, 0].imshow(denormalize(images[i]))
    axes[i, 0].set_title('Input Image')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(colorize_drivable(val_batch['drivable'][i].numpy()))
    axes[i, 1].set_title('GT Drivable Area')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(colorize_drivable(driv_pred[i]))
    axes[i, 2].set_title('Predicted Drivable Area')
    axes[i, 2].axis('off')

    axes[i, 3].imshow(depth_pred[i], cmap='magma')
    axes[i, 3].set_title('Predicted Depth')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Debug: Check loss scales on a single batch
model.eval()
batch = next(iter(val_loader))
images   = batch['image'].to(device)
driv_gt  = batch['drivable'].to(device)
depth_gt = batch['depth'].to(device)

with torch.no_grad():
    driv_pred, depth_pred = model(images)
    loss_driv_val  = drivable_criterion(driv_pred, driv_gt).item()
    loss_depth_val = depth_criterion(depth_pred, depth_gt).item()

print(f'\n=== Loss Scale Analysis ===')
print(f'Drivable Loss (DiceLoss):      {loss_driv_val:.6f}')
print(f'Depth Loss (InverseHuberLoss): {loss_depth_val:.6f}')
print(f'Total Loss:                    {loss_driv_val + loss_depth_val:.6f}')
print(f'\nRatios:')
print(f'  Depth / Drivable: {loss_depth_val / loss_driv_val:.3f}x')
print(f'  Drivable % of total: {100 * loss_driv_val / (loss_driv_val + loss_depth_val):.1f}%')
print(f'  Depth % of total:    {100 * loss_depth_val / (loss_driv_val + loss_depth_val):.1f}%')

if loss_depth_val / loss_driv_val > 1.5:
    print(f'\n⚠️  DEPTH loss is {loss_depth_val / loss_driv_val:.1f}x LARGER → depth predictions may be dominating')
elif loss_depth_val / loss_driv_val < 0.67:
    print(f'\n⚠️  DEPTH loss is {loss_driv_val / loss_depth_val:.1f}x SMALLER → drivable predictions may be dominating')
else:
    print(f'\n✓ Losses are reasonably balanced')

# 8 — Architecture Visualization

Visualize the full HydraNet computation graph with torchviz.

In [ ]:
from torchviz import make_dot
import torch

# Create a dummy input matching the model's expected input size
# requires_grad=True is needed so torchviz can trace the computation graph
dummy_input = torch.randn(1, 3, *INPUT_SIZE).to(device).requires_grad_(True)

# Run a forward pass WITHOUT torch.no_grad() — torchviz needs the gradient graph
model.eval()
output = model(dummy_input)

# If model returns multiple outputs (e.g., segmentation + depth), combine them
if isinstance(output, (tuple, list)):
    # Combine outputs into a single scalar for visualization
    combined = sum(o.sum() for o in output)
    graph = make_dot(combined, params=dict(model.named_parameters()))
else:
    graph = make_dot(output, params=dict(model.named_parameters()))

# Render and display the graph
graph.attr(rankdir='TB')
graph.render('hydranet_graph', format='png', cleanup=True)

from IPython.display import Image
Image('hydranet_graph.png')

In [ ]:
# Save model to Google Drive
import shutil

drive_path = '/content/gdrive/My Drive/hydranet_checkpoints'
os.makedirs(drive_path, exist_ok=True)

src = 'hydranet_best.pth'
dst = os.path.join(drive_path, 'hydranet_best.pth')
shutil.copy(src, dst)
print(f'Model saved to {dst}')
print(f'File size: {os.path.getsize(dst) / (1024**2):.1f} MB')

# 🎬 Video Inference — Drivable Area + Depth

BDD100K includes **100k dashcam video clips** (40s each, 720p, 30fps).
Upload one and watch HydraNet process it in real time!

In [ ]:
import cv2
from google.colab import files
from IPython.display import HTML
from base64 import b64encode

# Upload a BDD100K dashcam video
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
output_path = video_path.rsplit('.', 1)[0] + '_hydranet.mp4'

# Drivable overlay colors
VID_COLORS = np.array([[0,0,0], [0,180,0], [0,100,255]], dtype=np.uint8)

def overlay_drivable(frame, mask, alpha=0.4):
    color_mask = VID_COLORS[mask]
    out = frame.copy()
    active = mask > 0
    out[active] = cv2.addWeighted(frame[active], 1-alpha, color_mask[active], alpha, 0)
    return out

infer_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cap = cv2.VideoCapture(video_path)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {W}x{H} @ {fps:.0f} fps, {total} frames')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (W * 2, H))

model.eval()
processed = 0
t0 = time.time()

with torch.no_grad():
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        x = infer_transform(frame_rgb).unsqueeze(0).to(device)
        drv_out, dep_out = model(x)

        # Drivable overlay
        drv = drv_out.argmax(1).squeeze().cpu().numpy().astype(np.uint8)
        drv = cv2.resize(drv, (W, H), interpolation=cv2.INTER_NEAREST)
        left = overlay_drivable(frame, drv)

        # Depth colormap
        dep = dep_out.squeeze().cpu().numpy()
        dep = cv2.resize(dep, (W, H), interpolation=cv2.INTER_LINEAR)
        right = cv2.applyColorMap((dep * 255).astype(np.uint8), cv2.COLORMAP_MAGMA)

        writer.write(np.concatenate([left, right], axis=1))
        processed += 1
        if processed % 200 == 0:
            print(f'  {processed}/{total} frames — {processed/(time.time()-t0):.1f} fps')

cap.release()
writer.release()
elapsed = time.time() - t0
print(f'\nDone! {processed} frames in {elapsed:.1f}s ({processed/elapsed:.1f} fps)')

# Display in notebook
mp4 = open(output_path, 'rb').read()
b64 = b64encode(mp4).decode()
HTML(f'<video controls width="960"><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>')

# 9 — Save Pre-computed Features for Module 3

We save the backbone features for the validation set so students in Module 3 can train new heads **without running the backbone** — making training fast enough for a lab session.

In [ ]:
import os

os.makedirs('precomputed', exist_ok=True)
torch.save(model.state_dict(), 'precomputed/hydranet_module2.pth')

model.eval()
all_necks, all_feats0, all_filenames = [], [], []

print("Pre-computing features for Module 3...")
with torch.no_grad():
    for batch in tqdm(train_loader, desc='Pre-computing'):
        images = batch['image'].to(device)
        feats  = model.backbone(images)
        ctx    = model.drivable_context(feats[4])
        neck   = model.neck(ctx, feats)
        all_necks.append(neck.cpu())
        all_feats0.append(feats[0].cpu())
        all_filenames.extend(batch['filename'])

torch.save({
    'neck':       torch.cat(all_necks,  dim=0),
    'features_0': torch.cat(all_feats0, dim=0),
    'filenames':  all_filenames,
}, 'precomputed/train_features.pt')

print(f"Saved {len(all_filenames)} feature sets.")
print(f"Neck:       {torch.cat(all_necks, dim=0).shape}")
print(f"Features_0: {torch.cat(all_feats0, dim=0).shape}")
print(f"\nStudents can now train new heads in minutes!")

# Next Steps

In **Module 3**, you'll add new heads to this pre-trained HydraNet:
- **Lane Detection** — ego-lane segmentation (demo)
- **2D Object Detection** — anchor-free CenterNet-style heatmaps (lab)
- **Traffic Attribute Prediction** — weather / time of day classification (lab)

The backbone is frozen, so training new heads takes only **5-15 minutes** on Colab!